In [10]:
import os
from pathlib import Path
import pandas as pd
import tifffile as tiff
from skimage.measure import label, regionprops
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np

In [2]:
# Define paths
home_path = os.path.expanduser("~")
data_path = Path(home_path) / 'ext_hd_sammy' / 'data' / 'COMET'

image_path = data_path / 'original'
masks_comet_path = os.path.join(data_path, "visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples")
save_dir = Path(home_path) / 'ext_hd_sammy' / 'projects' / 'out' / 'out_comet'
save_dir.mkdir(parents=True, exist_ok=True)

# Markers (channels)
markers = [
    'DAPI', 'CY5', 'TRITC', 'CD4', 'CD31', 'CD8', 'CD45RO', 'CD163', 'FOXp3',
    'TOX1', 'CD20', 'CD86', 'CD68', 'CD10', 'CD133', 'CD11C', 'GZMB',
    'Periostin', 'CD66B', 'COL1A1', 'CD56', 'KERATIN8', 'CD45', 'aSMA'
]

# Parameters
batch_size = 900
n_jobs = 27  # Adjust based on system

In [3]:
def create_batches(props, batch_size):
    """Split regionprops into batches."""
    for i in range(0, len(props), batch_size):
        yield props[i:i + batch_size]


def extract_polygon_vertices(batch):
    """Extract polygon vertices for each labeled cell in the batch."""
    polygons = {}
    for prop in batch:
        lbl = prop.label
        coords = prop.coords
        ### sort by angle from centroid to ensure consistent ordering
        centroid = prop.centroid
        angles = np.arctan2(coords[:, 0] - centroid[0], coords[:, 1] - centroid[1])
        sort_idx = np.argsort(angles)
        coords = coords[sort_idx]
        
        ### close the polygon by appending the first point at the end
        coords = np.vstack([coords, coords[0]])
        
        polygons[lbl] = coords.tolist()
    return polygons

def extract_batch_protein_df(batch, orig_img, single_cell, markers):
    """Extract mean intensities for given markers from original image for each labeled cell."""
    
    df = pd.DataFrame(columns=markers + ['centroid_x', 'centroid_y'])

    # Check that image has at least as many channels as markers
    if orig_img.shape[0] < len(markers):
        raise ValueError(f"Image has only {orig_img.shape[0]} channels but {len(markers)} markers were given.")

    for prop in batch:
        centroid_y, centroid_x = prop.centroid
        lbl = prop.label
        mask = single_cell == lbl

        # Extract intensities for each marker channel
        protein_expression = []
        for i in range(len(markers)):
            ch_img = orig_img[i, ...]
            masked = ch_img[mask]
            mean_val = masked.mean() if masked.size > 0 else 0
            protein_expression.append(mean_val)

        # Append centroids
        protein_expression.extend([centroid_x, centroid_y])

        # Set row in dataframe using label as index
        df.loc[lbl] = protein_expression

    return df

In [4]:
def process_single_visio_object(visio_object, visio_out, image_path, orig_images, markers, save_dir, batch_size, n_jobs):
    """Process a single segmentation + image pair, and extract protein intensities."""
    base_name = visio_object.split('.tif')[0]
    print(f"\nProcessing {base_name}...")

    # Load segmentation mask
    seg_path = Path(visio_out) / visio_object
    seg_img = tiff.imread(seg_path)
    single_cell = label(seg_img[0, ...] > 0)
    props = regionprops(single_cell)

    # Load matching full image
    try:
        orig_img = next(
            tiff.imread(Path(image_path) / img)
            for img in orig_images if base_name in img
        )
    except StopIteration:
        print(f"[Warning] Original image not found for: {base_name}")
        return

    # Create batches
    batches = list(create_batches(props, batch_size))

    # Run parallel batch extraction
    #batch_dfs = Parallel(n_jobs=n_jobs, prefer="threads")(
    #    delayed(extract_batch_protein_df)(batch, orig_img, single_cell, markers)
    #    for batch in tqdm(batches, desc=f"🔬 Extracting {base_name}", leave=False)
    #)

    batch_vertices = Parallel(n_jobs=n_jobs, prefer="threads")(
        delayed(extract_polygon_vertices)(batch)
        for batch in tqdm(batches, desc=f"📐 Extracting polygons {base_name}", leave=False)
    )
    
        # Combine all polygon dictionaries into one
    combined_polygons = {}
    for poly_dict in batch_vertices:
        combined_polygons.update(poly_dict)

    # Convert to DataFrame
    # Each row will have: label, x_coords (list), y_coords (list)
    polygon_df = pd.DataFrame([
        {
            "label": lbl,
            "x_coords": [p[1] for p in coords],
            "y_coords": [p[0] for p in coords]
        }
        for lbl, coords in combined_polygons.items()
    ])

    # Save polygons as CSV
    out_poly_csv = Path(save_dir) / f"{base_name}_polygon_vertices.csv"
    polygon_df.to_csv(out_poly_csv, index=False)
    print(f"Saved: {out_poly_csv}")

    # Combine and save
    #full_df = pd.concat(batch_dfs, axis=0)
    #out_csv = Path(save_dir) / f"{base_name}_protein_combined.csv"
    #full_df.to_csv(out_csv, index=False)
    #print(f"Saved: {out_csv}")

In [5]:
# List all image/segmentation pairs
visio_objects = sorted([f for f in os.listdir(masks_comet_path) if f.endswith('.tif')])
orig_images = sorted([f for f in os.listdir(image_path) if f.endswith('.tif') or f.endswith('.tiff')])

visio_objects

['SO1_BS.tif',
 'SO20(Aligned)_BS.tif',
 'SO36_BS.tif',
 'SO39_BS.tif',
 'SO40_BS.tif',
 'SO48_BS.tif',
 'SO53_BS.tif',
 'SO64_BS.tif',
 'SO67_BS.tif',
 'SO6_BS.tif',
 'SO9_BS.tif']

In [6]:
image_path

PosixPath('/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/original')

In [7]:
visio_objects = visio_objects[-2:]

In [8]:
orig_images

['SO13_BS.ome.tiff',
 'SO14_BS.ome.tiff',
 'SO15_BS.ome.tiff',
 'SO1_BS.ome.tiff',
 'SO23_BS.ome.tiff',
 'SO24_BS.ome.tiff',
 'SO26_BS.ome.tiff',
 'SO2_BS.ome.tiff',
 'SO32_BS.ome.tiff',
 'SO33_BS.ome.tiff',
 'SO34_BS.ome.tiff',
 'SO44_BS.ome.tiff',
 'SO45_BS.ome.tiff',
 'SO46_BS.ome.tiff',
 'SO4_BS.ome.tiff',
 'SO50_BS.ome.tiff',
 'SO51_BS.ome.tiff',
 'SO58_BS.ome.tiff',
 'SO5_BS.ome.tiff',
 'SO6_BS.ome.tiff']

In [ ]:

# Run tile-level extraction
for visio_object in tqdm(visio_objects, desc="Processing all Visio objects"):
    process_single_visio_object(
        visio_object=visio_object,
        visio_out=masks_comet_path,
        image_path=image_path,
        orig_images=orig_images,
        markers=markers,
        save_dir=save_dir,
        batch_size=batch_size,
        n_jobs=n_jobs
    )

def merge_per_sample_and_cleanup(save_dir):
    """Merge per-tile files into one CSV per sample. Delete tile files. Do NOT make master CSV."""
    save_dir = Path(save_dir)
    csv_files = sorted(save_dir.glob("*_protein_combined.csv"))

    if not csv_files:
        print("No per-tile CSVs found.")
        return

    sample_dfs = {}

    for f in tqdm(csv_files, desc="🔗 Merging per-sample files"):
        tile_name = f.stem.replace("_protein_combined", "")
        sample_name = tile_name.split('_')[0]

        df = pd.read_csv(f)
        df["tile"] = tile_name
        df["sample"] = sample_name
        df["label"] = df.index

        sample_dfs.setdefault(sample_name, []).append(df)

    # Write one file per sample
    for sample, dfs in sample_dfs.items():
        merged = pd.concat(dfs, axis=0)
        merged.to_csv(save_dir / f"{sample}_protein_combined.csv", index=False)
        print(f"Saved: {sample}_protein_combined.csv")

    # Delete intermediate tile-level CSVs
    for f in csv_files:
        f.unlink()
    print(f"Deleted {len(csv_files)} tile CSV files.")

Processing all Visio objects:   0%|          | 0/2 [00:00<?, ?it/s]


Processing SO6_BS...




















Processing all Visio objects:  50%|█████     | 1/2 [15:54<15:54, 954.00s/it]

Saved: /home/shamini_pathomics_io/ext_hd_sammy/projects/out/out_comet/SO6_BS_polygon_vertices.csv

Processing SO9_BS...
